# GRACE H3-v2 — Bắt đầu full pipeline local

Branch: **feat/h3-v2-benchmark**. MCC là chính, F1 thứ hai; cấu hình mặc định Devign + ReVeal, seed42.

| Lượt | Việc làm | Tiếp tục ở đâu |
|---|---|---|
| 1 | Entry tạo base-inputs local | Chạy colab_cpu_prepare.ipynb trên máy, PHASE='base' |
| 2 | Entry nhập base-output, chạy GRACE validation | Chạy lại entry; sau đó runner local, PHASE='meta' |
| 3 | Entry nhập meta-output, chạy GRACE test và đánh giá | Xem benchmark_summary.json trong runs/h3v2/ |

Notebook CPU giữ tên cũ để không mất đường dẫn, nhưng chạy hoàn toàn local: không Colab, Drive, upload hay download. Mọi zip input/output/checkpoint nằm trong bundles/.

CPU stages chạy offline khi dependencies và Java21 đã cài sẵn. Vertex LLM cần mạng trừ khi exact response cache đủ mọi request. RUN_INFERENCE=True bật gọi API; False chỉ dùng cache.

H3-v2 chưa có benchmark. Test không dùng để chọn winner.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
candidates=[Path.cwd(),Path.cwd().parent,Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')]
if os.environ.get('GRACE_H3V2_ROOT'): candidates.insert(0,Path(os.environ['GRACE_H3V2_ROOT']))
ROOT=next((p.resolve() for p in candidates if (p/'configs/benchmark_v2.json').exists()),None)
if ROOT is None: raise RuntimeError('Mở notebook từ worktree GRACE-H3-BenchmarkEnsemble hoặc đặt GRACE_H3V2_ROOT')
sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=False
PREPARE_IN_LOCAL_NOTEBOOK=True
RUN_INFERENCE=True
BUNDLES=ROOT/'bundles'
BUNDLES.mkdir(exist_ok=True)
print('ROOT:',ROOT)
print('Entry local của branch benchmark; CPU runner chạy trên cùng máy.')


In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','--break-system-packages','-r',str(ROOT/'requirements-v2.txt'),'-r',str(ROOT/'requirements.txt')])
from grace_h1.io import load_local_environment
load_local_environment(ROOT)
from grace_h3v2 import workflow as w
from grace_h3v2.portable import create_bundle,import_bundle,identity
import zipfile
import pandas as pd
CFG=w.config(ROOT)
print(json.dumps(CFG,indent=2))
def show_stage(name):
    try: return w.load_stage(ROOT,CFG,name)
    except FileNotFoundError: return None
def cpu_handoff(phase,selection):
    output=BUNDLES/f'grace-h3v2-{phase}-output.zip'
    if output.exists(): print(import_bundle(ROOT,output,'output',phase))
    result=show_stage(selection)
    if result is not None: return result
    if not PREPARE_IN_LOCAL_NOTEBOOK:
        return w.base_prepare(ROOT,CFG) if phase=='base' else w.meta_prepare(ROOT,CFG)
    archive=BUNDLES/f'grace-h3v2-{phase}-inputs.zip'
    reuse=False
    if archive.exists():
        with zipfile.ZipFile(archive) as z:
            m=json.loads(z.read('bundle_manifest.json'))
            reuse=m.get('job')==identity(ROOT) and m.get('phase')==phase and m.get('kind')=='inputs'
    if not reuse: print(create_bundle(ROOT,archive,'inputs',phase))
    raise RuntimeError(f'DỪNG: mở notebooks/colab_cpu_prepare.ipynb trên máy local, đặt PHASE={phase!r} và Run All. Input: {archive}. Output sẽ ghi vào {output}. Sau đó quay lại notebook này và Run All.')


## Bước 1 — CPU local: classifier và chuẩn bị validation

Entry tạo base-inputs rồi dừng. Chạy notebooks/colab_cpu_prepare.ipynb local với PHASE='base'. CPU fit classifier bằng nested train OOF, chọn gate bằng MCC/F1, chuẩn bị prompts validation và ghi base-output vào bundles/. CodeT5 frozen; CPU runner không gọi LLM.


In [ ]:
base=w.base_prepare(ROOT,CFG) if not PREPARE_IN_LOCAL_NOTEBOOK else cpu_handoff('base','base_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'base_folder':j['base_folder']} for j in base['jobs']]))


## Bước 2 — Local: GRACE validation ngoài test

Preflight và gọi các request còn thiếu tại local; exact cache được reuse. API cần mạng. Khi validation complete, Run All tạo meta-inputs local cho lượt CPU tiếp theo.


In [ ]:
plans=w.preflight(ROOT,CFG,'validation')
print(json.dumps(plans,indent=2))
reports=w.infer(ROOT,CFG,'validation',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Validation chưa complete: bật RUN_INFERENCE hoặc xử lý API/ADC rồi chạy lại. Chưa chuyển meta-fit.')


## Bước 3 — CPU local: meta selector/stacking và test features

Sau validation complete, entry tạo meta-inputs trong bundles/. Chạy colab_cpu_prepare.ipynb local với PHASE='meta'. CPU fit router/selector/stacking trên development, khóa winner, tính test features không nhãn, và xuất meta-output vào bundles/. Không gọi LLM ở CPU runner.

Nhánh benefit-router chỉ dùng feature trước expert; selector/stacking dùng expert output. Full-reference/stacking có thể cần100% query: benchmark mode.


In [ ]:
meta=w.meta_prepare(ROOT,CFG) if not PREPARE_IN_LOCAL_NOTEBOOK else cpu_handoff('meta','meta_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'winner_development':j['development_winner']} for j in meta['jobs']]))


## Bước 4 — Local: test inference và đánh giá

Winner đã khóa từ development. Local preflight/test-infer reuse exact cache hoặc gọi Vertex nếu RUN_INFERENCE=True. Test scores đã tính ở CPU runner local; nhãn test chỉ mở trong evaluate.


In [ ]:
print(json.dumps(w.preflight(ROOT,CFG,'test'),indent=2))
reports=w.infer(ROOT,CFG,'test',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Test requests chưa complete; resume inference trước evaluate.')
w.predict(ROOT,CFG)
summary=w.evaluate(ROOT,CFG)


In [ ]:
table=[]
for run in summary['datasets']:
    for arm,result in run['arms'].items():
        m=result['metrics']
        table.append({'dataset':run['dataset'],'seed':run['seed'],'arm':arm,'locked_winner':arm==run['locked_development_winner'],'MCC':m['mcc'],'F1':m['f1'],'precision':m['precision'],'recall':m['recall'],'specificity':m['specificity'],'invalid':m['invalid_n'],'expert_query_ratio':result['algorithmic_expert_query_ratio'],'fallback':result['fallback_n']})
display(pd.DataFrame(table))
print('Summary:',w.stage_file(ROOT,CFG,'benchmark_summary'))
print('Không chọn lại winner dựa trên bảng test. So với gate_tuned và full_grace; chi phí routing lý thuyết tách actual API/cache.')
